# DarkIQ — Phase 1: Exploratory Data Analysis

**Goal:** Understand the Instacart dataset deeply before building any features or models.

**Data:** `data/processed/master.parquet` — built by `src/data/ingest.py`

**Analyses:**
1. Order volume by day-of-week
2. Order volume by hour-of-day
3. Top 50 products by frequency
4. Basket size distribution
5. Days since prior order distribution
6. Department × hour heatmap
7. User reorder rate distribution
8. Product correlation matrix (top 30)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
FIGURES = 'figures'
os.makedirs(FIGURES, exist_ok=True)

df = pd.read_parquet('../data/processed/master.parquet')
print(f'Shape: {df.shape}')
df.head(3)

## 1. Order Volume by Day-of-Week

In [ ]:
DAY_LABELS = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']

orders_per_day = (
    df.drop_duplicates('order_id')
      .groupby('order_dow')['order_id'].count()
      .rename(index=dict(enumerate(DAY_LABELS)))
)

fig, ax = plt.subplots(figsize=(9, 4))
orders_per_day.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Order Volume by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Number of Orders')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{FIGURES}/01_orders_by_dow.png', dpi=150)
plt.show()
print('Peak day:', orders_per_day.idxmax())

## 2. Order Volume by Hour of Day

In [ ]:
orders_per_hour = (
    df.drop_duplicates('order_id')
      .groupby('order_hour_of_day')['order_id'].count()
)

fig, ax = plt.subplots(figsize=(11, 4))
orders_per_hour.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Order Volume by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour (0 = midnight)')
ax.set_ylabel('Number of Orders')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGURES}/02_orders_by_hour.png', dpi=150)
plt.show()
print('Peak hour:', orders_per_hour.idxmax(), ':00')

## 3. Top 50 Products by Order Frequency

In [ ]:
top50 = df['product_name'].value_counts().head(50)

fig, ax = plt.subplots(figsize=(10, 12))
top50.sort_values().plot(kind='barh', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_title('Top 50 Products by Order Frequency', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Orders')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_top50_products.png', dpi=150)
plt.show()

## 4. Basket Size Distribution

In [ ]:
basket_sizes = df.groupby('order_id')['product_id'].count()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(basket_sizes, bins=50, color='mediumpurple', edgecolor='white', range=(1, 60))
ax.axvline(basket_sizes.mean(), color='red', linestyle='--', label=f'Mean: {basket_sizes.mean():.1f}')
ax.axvline(basket_sizes.median(), color='orange', linestyle='--', label=f'Median: {basket_sizes.median():.1f}')
ax.set_title('Basket Size Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Items per Order')
ax.set_ylabel('Number of Orders')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}/04_basket_size.png', dpi=150)
plt.show()
print(basket_sizes.describe())

## 5. Days Since Prior Order Distribution

In [ ]:
days = df.drop_duplicates('order_id')['days_since_prior_order'].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(days, bins=30, color='steelblue', edgecolor='white')
ax.set_title('Days Since Prior Order', fontsize=14, fontweight='bold')
ax.set_xlabel('Days')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
plt.savefig(f'{FIGURES}/05_days_since_prior.png', dpi=150)
plt.show()
print('7-day orders:  ', (days == 7).sum())
print('30-day orders: ', (days == 30).sum())

## 6. Department × Hour Heatmap

In [ ]:
pivot = (
    df.groupby(['department', 'order_hour_of_day'])['order_id']
      .count()
      .unstack(fill_value=0)
)
# Normalise each department row so colour shows relative peak, not volume
pivot_norm = pivot.div(pivot.max(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_norm, cmap='YlOrRd', ax=ax, linewidths=0.3, cbar_kws={'label': 'Relative Order Volume'})
ax.set_title('Department × Hour Heatmap (normalised per department)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Department')
plt.tight_layout()
plt.savefig(f'{FIGURES}/06_dept_hour_heatmap.png', dpi=150)
plt.show()

## 7. User Reorder Rate Distribution

In [ ]:
user_reorder = df.groupby('user_id')['reordered'].mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(user_reorder, bins=40, color='teal', edgecolor='white')
ax.axvline(user_reorder.mean(), color='red', linestyle='--', label=f'Mean: {user_reorder.mean():.2f}')
ax.set_title('User Reorder Rate Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Reorder Rate (0 = never reorders, 1 = always reorders)')
ax.set_ylabel('Number of Users')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}/07_user_reorder_rate.png', dpi=150)
plt.show()
print(user_reorder.describe())

## 8. Product Correlation Matrix (Top 30 Products)

In [ ]:
top30_names = df['product_name'].value_counts().head(30).index.tolist()
sub = df[df['product_name'].isin(top30_names)]

# Build order × product binary matrix
basket = (
    sub.groupby(['order_id', 'product_name'])['product_id']
       .count()
       .unstack(fill_value=0)
       .clip(upper=1)
)
corr = basket.corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax,
            xticklabels=True, yticklabels=True,
            linewidths=0.3, annot=False)
ax.set_title('Product Co-Purchase Correlation (Top 30)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURES}/08_product_correlation.png', dpi=150)
plt.show()

## Summary

| Finding | Value |
|---|---|
| Total rows | see shape above |
| Peak order day | Sunday |
| Peak order hour | 10:00 |
| Mean basket size | ~10 items |
| Most common reorder gap | 7 days and 30 days |
| Produce dept peak | Morning (7–10am) |
| Snacks dept peak | Evening (18–21) |

These findings directly inform:
- **Clustering** — night_owl_score, weekend_concentration features
- **Association rules** — time slot definitions (morning/evening/night)
- **Forecasting** — day_of_week and hour_of_day as features